# Day3_15 · 랜덤포레스트가 사용한 정보 확인

평가자료에서 입력열을 섞고 예측 변화를 확인해 랜덤포레스트를 해석합니다.

## 이 노트북에서 확인할 내용

- 평가자료에서 순열 중요도를 계산할 수 있습니다.
- 두 종류의 변수 중요도가 다른 이유를 설명할 수 있습니다.
- 부분의존도와 인과효과를 구분할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-15-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-15-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-15-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-15-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-15-05 · pandas 불러오기

In [ ]:
import pandas as pd

### D3B-15-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-15-07 · 콘크리트 강도자료 읽기

In [ ]:
concrete = pd.read_csv(DATA_DIR / 'concrete_strength.csv')

In [ ]:
concrete.head()

### D3B-15-08 · 입력 열 정하기

In [ ]:
feature_columns = ['cement_kg_m3', 'slag_kg_m3', 'fly_ash_kg_m3', 'water_kg_m3', 'superplasticizer_kg_m3', 'coarse_aggregate_kg_m3', 'fine_aggregate_kg_m3', 'age_days']

In [ ]:
pd.Series(feature_columns)

### D3B-15-09 · 입력표 만들기

In [ ]:
X = concrete[feature_columns].astype(float)

In [ ]:
X.head()

### D3B-15-10 · 결과 열 만들기

In [ ]:
y = concrete['strength_mpa']

In [ ]:
y.head()

### D3B-15-11 · 자료 분할 함수 불러오기

In [ ]:
from sklearn.model_selection import train_test_split

### D3B-15-12 · 학습자료와 평가자료 나누기

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
pd.Series({'train_rows': len(X_train), 'test_rows': len(X_test)})

### D3B-15-13 · 랜덤포레스트 불러오기

In [ ]:
from sklearn.ensemble import RandomForestRegressor

### D3B-15-14 · 랜덤포레스트 학습하기

In [ ]:
forest_model = RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1).fit(X_train, y_train)

In [ ]:
forest_model

### D3B-15-15 · 평가자료 예측하기

In [ ]:
forest_prediction = forest_model.predict(X_test)

In [ ]:
pd.Series(forest_prediction[:5], name='predicted_strength_mpa')

### D3B-15-16 · RMSE 도구 불러오기

In [ ]:
from sklearn.metrics import root_mean_squared_error

### D3B-15-17 · 평가자료 RMSE 계산

In [ ]:
forest_rmse = root_mean_squared_error(y_test, forest_prediction)

In [ ]:
forest_rmse

## 입력열 하나를 섞어서 모델의 의존성 확인하기

평가자료의 한 열만 무작위로 섞은 뒤 RMSE가 얼마나 나빠지는지 확인합니다. 많이 나빠진다면 현재 모델이 그 열의 정보에 상대적으로 많이 의존했다는 뜻입니다.

### D3B-15-18 · 순열 중요도 도구 불러오기

In [ ]:
from sklearn.inspection import permutation_importance

### D3B-15-19 · 평가자료에서 순열 중요도 계산

In [ ]:
permutation_result = permutation_importance(
    forest_model, X_test, y_test,
    scoring='neg_root_mean_squared_error',
    n_repeats=20, random_state=42, n_jobs=-1,
)

In [ ]:
permutation_result

### D3B-15-20 · 순열 중요도 표 만들기

In [ ]:
permutation_table = pd.DataFrame({
    'input_column': feature_columns,
    'RMSE_increase_mean': permutation_result.importances_mean,
    'RMSE_increase_std': permutation_result.importances_std,
}).sort_values('RMSE_increase_mean', ascending=False)

In [ ]:
permutation_table.round(3)

### D3B-15-21 · 순열 중요도 막대그래프

In [ ]:
plot_table = permutation_table.sort_values('RMSE_increase_mean')
plt.barh(plot_table['input_column'], plot_table['RMSE_increase_mean'], xerr=plot_table['RMSE_increase_std'])
plt.xlabel('Increase in RMSE after shuffling')
plt.show()

음수에 가까운 값은 그 열이 보호효과를 낸다는 뜻이 아닙니다. 반복된 섞기, 표본의 우연성, 서로 비슷한 정보를 가진 입력열 때문에 성능이 거의 변하지 않거나 조금 좋아질 수 있습니다.

### D3B-15-22 · 나무 내부 중요도 표 만들기

In [ ]:
tree_importance_table = pd.DataFrame({
    'input_column': feature_columns,
    'tree_importance': forest_model.feature_importances_,
}).sort_values('tree_importance', ascending=False)

In [ ]:
tree_importance_table.round(3)

### D3B-15-23 · 두 중요도 순위 비교

In [ ]:
importance_comparison = permutation_table.merge(tree_importance_table, on='input_column')

In [ ]:
importance_comparison.round(3)

나무 내부 중요도와 평가자료 순열 중요도는 계산 질문이 다르므로 순위가 같지 않을 수 있습니다. 연구 보고에서는 평가자료에서 계산한 순열 중요도를 중심으로 설명하고 모델과 점수를 함께 기록합니다.

### D3B-15-24 · 부분의존도 그림 도구 불러오기

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

### D3B-15-25 · 재령의 부분의존도 그리기

In [ ]:
PartialDependenceDisplay.from_estimator(forest_model, X_test, ['age_days'], kind='average')
plt.ylabel('Average predicted strength (MPa)')
plt.show()

### D3B-15-26 · 물 사용량의 부분의존도 그리기

In [ ]:
PartialDependenceDisplay.from_estimator(forest_model, X_test, ['water_kg_m3'], kind='average')
plt.ylabel('Average predicted strength (MPa)')
plt.show()

부분의존도는 다른 입력값을 평가자료에 있는 상태로 두고 한 입력값을 바꾸었을 때 모델의 평균 예측이 어떻게 달라지는지 보여 줍니다. 실제 배합을 바꾼 실험 결과나 인과효과가 아닙니다.

### D3B-15-27 · 평가 결과표 만들기

In [ ]:
error_table = X_test.copy()
error_table['actual_mpa'] = y_test
error_table['predicted_mpa'] = forest_prediction
error_table['absolute_error_mpa'] = (error_table['actual_mpa'] - error_table['predicted_mpa']).abs()

In [ ]:
error_table.head().round(2)

### D3B-15-28 · 오차가 큰 평가 행 확인

In [ ]:
largest_forest_errors = error_table.nlargest(10, 'absolute_error_mpa')

In [ ]:
largest_forest_errors.round(2)

### D3B-15-29 · 부분의존도를 볼 열 선택

In [ ]:
# age_days를 cement_kg_m3 또는 slag_kg_m3로 바꾸어 보세요.
selected_feature = 'cement_kg_m3'

In [ ]:
selected_feature

### D3B-15-30 · 선택한 열의 부분의존도 그리기

In [ ]:
PartialDependenceDisplay.from_estimator(forest_model, X_test, [selected_feature], kind='average')
plt.ylabel('Average predicted strength (MPa)')
plt.show()

### D3B-15-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''forest_model, X_test, y_test, feature_columns를 그대로 사용하세요. 기존 셀은 수정하지 마세요. 순열 중요도를 30회 반복해 표와 가로 막대그래프로 보여 주는 새 셀을 제안해 주세요. 중요도를 원인효과로 부르지 말고, 상관된 입력열이 있을 때 순위가 낮아질 수 있다는 주의 문장을 포함해 주세요.'''.strip()

In [ ]:
codex_request

### D3B-15-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['평가자료에서 중요도를 계산하는가?', '사용한 점수가 명시되는가?', '중요도를 인과효과로 표현하지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist